In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv('../Data/ml_models_results/temporal_comp_with_dbn.csv')

# Inspect the data
print(df.head())
print(df.info())
print(df['Target_Year'].unique())
print(df['Metric'].unique())
print(df['Model'].unique())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Clean Target_Year to just the year number for plotting
df['Year'] = df['Target_Year'].str.extract('(\d+)').astype(int)

# Create a column for formatted strings (Mean [Lower - Upper])
df['Formatted'] = df.apply(lambda row: f"{row['Mean']:.3f} ({row['95% CI Lower']:.3f}-{row['95% CI Upper']:.3f})", axis=1)

# Function to plot
def plot_metric(metric_name, title, filename, ylabel):
    subset = df[df['Metric'] == metric_name].sort_values(['Model', 'Year'])
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=subset, x='Year', y='Mean', hue='Model', marker='o', linewidth=2.5)
    plt.title(title, fontsize=16)
    plt.xlabel('Outcome Year', fontsize=14)
    plt.ylabel(ylabel, fontsize=14)
    plt.xticks(range(1, 7))
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

# Generate plots
plot_metric('roc_auc', 'Temporal Model Performance UCLA+PHS: ROC AUC', 'roc_auc_temporal.png', 'ROC AUC')
plot_metric('average_precision', 'Temporal Model Performance UCLA+PHS: Average Precision', 'average_precision_temporal.png', 'Average Precision')

# Create a wide table for the paper
table_df = df.pivot_table(index=['Year', 'Metric'], columns='Model', values='Formatted', aggfunc='first')
table_df.to_csv('formatted_results_table.csv')
print(table_df)

In [ ]:
# 1. Clean the Year column for sorting and display
# This extracts the numeric part (1, 2, 3...) from 'year1_reduction_40_ge'
df['Year'] = df['Target_Year'].str.extract('(\d+)').astype(int)

# 2. Format the string: Mean (95% CI Lower - 95% CI Upper)
# We round to 3 decimal places for scientific precision
df['Value'] = df.apply(
    lambda x: f"{x['Mean']:.2f} ({x['95% CI Lower']:.2f}-{x['95% CI Upper']:.2f})", 
    axis=1
)

# 3. Pivot the table to Wide Format
# Rows: Year and Metric | Columns: Model names
table_paper = df.pivot_table(
    index=['Year', 'Metric'], 
    columns='Model', 
    values='Value', 
    aggfunc='first'
)

# 4. Reorder and sort indices
# We sort by Year (ascending) and Metric (ROC AUC followed by Average Precision)
table_paper = table_paper.sort_index(level=[0, 1], ascending=[True, False])

# 5. Export to CSV for use in Word, Excel, or LaTeX
table_paper.to_csv('publication_ready_table.csv')

# Display the resulting table
print(table_paper)